# Fully-Connected Neural Networks

In this assignment you will implement the core operations of a small fully connected  neural network for image classification and use them to build the forward and
backward passes needed for training.

## Grading

There are five short coding tasks in
this notebook; each one corresponds to a function whose body you must
complete.  Every task includes a simple self-test so you can verify your
code before submission.
The table shows
the name of the function and how many points that task is worth.
Make sure
not to rename any of the functions – the autograder looks for these exact
names.

| Task | Submitted Function | Points |
|:-----|:-------------------|:------:|
| Task 1 | [`affine_forward`](#task-affine-layer-forward) | 5 |
| Task 2 | [`relu_forward`](#task-relu-activation-forward) | 5 |
| Task 3 | [`relu_backward`](#task-relu-activation-backward) | 5 |
| Task 4 | [`affine_backward`](#task-affine-layer-backward) | 15 |
| Task 5 | [`cross_entropy_loss`](#task-cross-entropy-loss) | 10 |
|  | **Total Points** | **40** |


## Google Colab 

In [ ]:
# NOTE: Execute this cell once before running any other code.
# It will install the necessary libraries and download the data.

# Install libraries in Colab
!pip install -q matplotlib numpy opencv-python

# Download additonal utility files and data
import os

if not os.path.exists("data"):
    !git clone https://github.com/hpi-mlia-2026/coding_assignments.git
    %cd coding_assignments/assignment_02

## Library imports

In [ ]:
from utils import *
import matplotlib.pyplot as plt
from builtins import range
from builtins import object

import numpy as np
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import Subset

<a id="Image_classification"></a>
## Image classification on CIFAR10
The task for this practical is to build a multi-class image classification model. The dataset we'll work with is the CIFAR10 image classification benchmark. It consists of 50,000 training images (each in $32 \times 32$ resolution with RGB color channels). Each pixel is represented as three floating point numbers, indicating the color intensity in the respective color channel. Each image is labeled with integers ranging from 0 to 9, indicating the class of the image content. The 10 classes and according example images from the dataset are shown in the image below. The dataset also consists of a test dataset, that consists of another 10,000 images and their labels. 

![Cifar10](https://pytorch.org/tutorials/_images/cifar10.png)

We further split the training images into 45,000 training samples and 5,000 validation samples. The network will just be trained on the training dataset and we use the validation data to check during training if the network generalizes well to new data. In practice, the validation dataset is used to tweak the hyperparameters of the model. The test data is not used until the very end of the process, to evaluate the models generalization ability on prior unseen data (which serves as a proxy for deploying the model in the real world).  

In [ ]:
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Build classical PyTorch CIFAR-10 datasets.
full_train_dataset = CIFAR10(root="./data", train=True, download=True, transform=cifar_transform)
test_dataset = CIFAR10(root="./data", train=False, download=True, transform=cifar_transform)

num_training = 45000
num_validation = 5000
train_dataset = Subset(full_train_dataset, list(range(num_training)))
val_dataset = Subset(full_train_dataset, list(range(num_training, num_training + num_validation)))


def get_CIFAR10_data(
    data_dir: str = "./data",
    num_training: int = 45000,
    num_validation: int = 5000,
    subtract_mean: bool = True,
):
    """
    Load CIFAR-10 using torchvision and return preprocessed numpy arrays.
    - Keeps deterministic split: first `num_training` of train set are training,
      next `num_validation` are validation.
    - Returns arrays in channel-first order (N, C, H, W), dtype float32.
    - Labels are 1-D numpy arrays of ints.
    """
    train_ds = CIFAR10(root=data_dir, train=True, download=True)
    test_ds = CIFAR10(root=data_dir, train=False, download=True)

    X_all = train_ds.data.astype(np.float32)
    y_all = np.array(train_ds.targets, dtype=np.int64)
    X_test = test_ds.data.astype(np.float32)
    y_test = np.array(test_ds.targets, dtype=np.int64)

    if num_training + num_validation > X_all.shape[0]:
        raise ValueError("Requested more training+validation samples than available in CIFAR-10 train split.")

    X_train = X_all[:num_training].copy()
    y_train = y_all[:num_training].copy()
    X_val = X_all[num_training : num_training + num_validation].copy()
    y_val = y_all[num_training : num_training + num_validation].copy()

    if subtract_mean:
        mean_image = np.mean(X_train, axis=0)
        X_train -= mean_image
        X_val -= mean_image
        X_test -= mean_image

    X_train = X_train.transpose(0, 3, 1, 2).astype(np.float32).copy()
    X_val = X_val.transpose(0, 3, 1, 2).astype(np.float32).copy()
    X_test = X_test.transpose(0, 3, 1, 2).astype(np.float32).copy()

    return {
        "X_train": X_train,
        "y_train": y_train.squeeze(),
        "X_val": X_val,
        "y_val": y_val.squeeze(),
        "X_test": X_test,
        "y_test": y_test.squeeze(),
    }


data = get_CIFAR10_data()

for k, v in data.items():
    if isinstance(v, (np.ndarray, np.generic)):
        print(f"{k}: {v.shape}, dtype={v.dtype}")
    else:
        print(f"{k}: {type(v).__name__}")


Let's look at a few training samples and their corresponding labels.

In [ ]:
# View samples and labels
class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(f"Class {label} ({class_names[label]})")
    ax.axis("off")

plt.suptitle('First 9 CIFAR-10 training samples with integer and semantic labels', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

<a id="fcnn"></a>
# 1. Implementing a Fully‑Connected Neural Net

We will construct a simple feed‑forward network with one hidden layer and an
output layer.  Both layers are fully connected to the previous layer; the
hidden layer uses a ReLU activation, and the output layer produces raw
scores (logits) for each class.

### 1.1 Forward pass

The diagram below shows the forward computation.  Input images have size
$32\times32\times3$, so when flattened each sample provides $D=3072$ input
features.  The first weight matrix $W^{(1)}$ therefore has shape $(D, H)$
where $H$ is the number of hidden units, and the second weight matrix
$W^{(2)}$ has shape $(H, 10)$ (ten output classes).

For layer $\ell$ we write

$$z^{(\ell)} = a^{(\ell-1)} W^{(\ell)} + b^{(\ell)},$$
$$a^{(\ell)} = g(z^{(\ell)}),$$

where $g$ is the activation function (ReLU for the hidden layer, identity
for the output layer).  The final activations $a^{(2)}$ are the logits of
our classifier; they are passed to a loss function $L_\theta$ that measures
the discrepancy with the true label.

![Network architecture](https://i.imgur.com/yv1FR86.png)

### Task: Affine layer (forward)
Now please implement the function `affine forward` below, which calculates pre-activations $z^{(l)} = w^{(l-1)}x + b^{(l)}$  and test your implementation with the provided test cases.
<a id="affine_forward"></a>

In [ ]:
def affine_forward(x, w, b):
    """
    Computes the forward pass for an affine (fully-connected) layer.

    The input x has shape (N, d_1, ..., d_k) and contains a minibatch of N
    examples, where each example x[i] has shape (d_1, ..., d_k). We will
    reshape each input into a vector of dimension D = d_1 * ... * d_k, and
    then transform it to an output vector of dimension M.

    Inputs:
    - x: A numpy array containing input data, of shape (N, d_1, ..., d_k)
    - w: A numpy array of weights, of shape (D, M)
    - b: A numpy array of biases, of shape (M,)

    Returns a tuple of:
    - out: output, of shape (N, M)
    - cache: (x, w, b)
    """
    out = None
    ###########################################################################
    # TODO: Implement the affine forward pass. Store the result in out. You   #
    # will need to reshape the input into rows.                               #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    out = None
    
    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ###########################################################################
    #                             END OF YOUR CODE                            #
    ###########################################################################
    cache = (x, w, b)
    return out, cache

In [ ]:
# Test the affine_forward function

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * np.prod(input_shape)
weight_size = output_dim * np.prod(input_shape)

x = np.linspace(-0.1, 0.5, num=input_size).reshape(num_inputs, *input_shape)
w = np.linspace(-0.2, 0.3, num=weight_size).reshape(np.prod(input_shape), output_dim)
b = np.linspace(-0.3, 0.1, num=output_dim)

out, _ = affine_forward(x, w, b)
correct_out = np.array([[ 1.49834967,  1.70660132,  1.91485297],
                        [ 3.25553199,  3.5141327,   3.77273342]])

# Compare your output with ours. The error should be around e-9 or less.
print('Testing affine_forward function:')
print('difference: ', rel_error(out, correct_out))

### Task: ReLU activation (forward)
Implement the forward pass for the ReLU activation function in the `relu_forward` function and test your implementation. The ReLU activation function is just $g(z) =  \begin{cases}
    z, & \text{for } z > 0 \\
    0, & \text{for } z \leq 0 \\
  \end{cases}$
<a id="relu_forward"></a>

In [ ]:
def relu_forward(x):
    """
    Computes the forward pass for a layer of rectified linear units (ReLUs).

    Input:
    - x: Inputs, of any shape

    Returns a tuple of:
    - out: Output, of the same shape as x
    - cache: x
    """
    out = None
    ###########################################################################
    # TODO: Implement the ReLU forward pass.                                  #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    out = None

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ###########################################################################
    #                             END OF YOUR CODE                            #
    ###########################################################################
    cache = x
    return out, cache


In [ ]:
# Test the relu_forward function

x = np.linspace(-0.5, 0.5, num=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = np.array([[ 0.,          0.,          0.,          0.,        ],
                        [ 0.,          0.,          0.04545455,  0.13636364,],
                        [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]])

# Compare your output with ours. The error should be on the order of e-8
print('Testing relu_forward function:')
print('difference: ', rel_error(out, correct_out))

## 1.2 Backward pass
The backward pass of the network is denoted in red. The general idea of the backpropagation of errors algorithm is to calculate partial derivatives of the loss $L_\theta$ with respect to the (pre-)activations of every layer in the network. Then the gradient descent algorithm is used to calculate the partial derivatives of the pre-activations with respect to the weights and biases.  

![Network architecture](https://i.imgur.com/SHe3l14.png) 

This procedure starts with the derivative of the loss with respect to the hypothesis $h_\theta(x)=a^{(3)}$, so $\frac{\partial L_{\theta}}{\partial a^{(3)}}$, which will be calculated by the function `cross_entropy_loss`. Then the derivative of the activation function $\frac{\partial a^{(l)}}{z^{(l)}}$ with respect to the pre-activations is calculated by the function `relu_backward` and we get $\delta^{(l)}=\frac{\partial L_{\theta}} {\partial a^{(l)}}\frac{\partial a^{(l)}}{z^{(l)}}$. Finally, the function `affine_backward` computes the derivatives `dw`$=\frac{\partial L_\theta}{\partial z^{(l)}} \frac{\partial z^{(l)}}{\partial w^{(l-1)}}$, `db`$=\frac{\partial L_\theta}{\partial z^{(l)}} \frac{\partial z^{(l)}}{\partial b^{(l-1)}}$ and `dx`$=\frac{\partial L_\theta}{\partial z^{(l)}} \frac{\partial z^{(l)}}{\partial a^{(l-1)}}$, which are needed to update the parameters of the model. 

Hint: When figuring out the derivatives needed for the back-propagation algorithm, you can always think about them in terms of the circuit notation used in the stanford cs231n course to check if they are plausible to you:

![Circuit](https://i.imgur.com/YN6z5gK.png)



### Task: ReLU activation (backward)
Now implement the backward pass for the ReLU activation function in the `relu_backward` function and test your implementation using gradient checking.
<a id="relu_backward"></a>

In [ ]:
def relu_backward(dout, cache):
    """
    Computes the backward pass for a layer of rectified linear units (ReLUs).

    Input:
    - dout: Upstream derivatives, of any shape
    - cache: Input x, of same shape as dout

    Returns:
    - dx: Gradient with respect to x
    """
    dx, x = None, cache
    ###########################################################################
    # TODO: Implement the ReLU backward pass.                                 #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    dx = None

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ###########################################################################
    #                             END OF YOUR CODE                            #
    ###########################################################################
    return dx


### Small excursion: Numeric gradient checking
Numeric gradient checking calculates a numerical approximation of the gradient by adding and substracting a small $\epsilon$ from the variable of interest. When we want to check the gradient of the weights $\theta$, we assume

$ \theta^{(i+)} = \theta + \begin{bmatrix} 0 \\  \vdots \\ \epsilon \\ \vdots \\  0 \end{bmatrix}$ and $ \theta^{(i-)} = \theta - \begin{bmatrix} 0 \\  \vdots \\ \epsilon \\ \vdots \\  0 \end{bmatrix}$

So, $\theta^{(i+)}$ is the same as $\theta$, except its $i^{th}$ element has been incremented by $\epsilon$. Similarly, $\theta^{(i−)}$ is the corresponding vector with the $i^{th}$ element decreased by $\epsilon$. You can now numerically verify $\frac{\partial L_{\theta}}{\partial \theta}$’s correctness by checking, for each $i$, that:

$$ \frac{\partial L_{\theta}^i}{\partial \theta} \approx \frac{L_{\theta^{(i+)}} - L_{\theta^{(i-)}}}{2\epsilon} $$

The degree to which these two values should approximate each other will depend on the details of $L_{\theta}$. But assuming $\epsilon = 10^{-8}$, you’ll usually find that the left- and right-hand sides of the above will agree to at least 8 significant digits (and often many more). This is already implemented in `eval_numerical_gradient_array`.

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10)
dout = np.random.randn(*x.shape)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

# The error should be on the order of e-12
print('Testing relu_backward function:')
print('dx error: ', rel_error(dx_num, dx))

### Task: Affine layer (backward)

You've already implemented `relu_backward`, and this function gets it's output $\frac{\partial L_\theta}{\partial z^{(l)}}$ as the input, together with the cached variables from the forward pass $a^{(l-1)}$, $w^{(l-1)}$ and $b^{(l-1)}$. It computes the derivatives `dw`$=\frac{\partial L_\theta}{\partial z^{(l)}} \frac{\partial z^{(l)}}{\partial w^{(l-1)}}$, `db`$=\frac{\partial L_\theta}{\partial z^{(l)}} \frac{\partial z^{(l)}}{\partial b^{(l-1)}}$ and `dx`$=\frac{\partial L_\theta}{\partial z^{(l)}} \frac{\partial z^{(l)}}{\partial a^{(l-1)}}$

Now implement the `affine_backward` function and test your implementation using numeric gradient checking.
<a id="affine_backward"></a>

In [ ]:
def affine_backward(dout, cache):
    """
    Computes the backward pass for an affine layer.

    Inputs:
    - dout: Upstream derivative, of shape (N, M)
    - cache: Tuple of:
      - x: Input data, of shape (N, d_1, ... d_k)
      - w: Weights, of shape (D, M)
      - b: Biases, of shape (M,)

    Returns a tuple of:
    - dx: Gradient with respect to x, of shape (N, d1, ..., d_k)
    - dw: Gradient with respect to w, of shape (D, M)
    - db: Gradient with respect to b, of shape (M,)
    """
    x, w, b = cache
    dx, dw, db = None, None, None
    ###########################################################################
    # TODO: Implement the affine backward pass.                               #
    ###########################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    dx = None
    dw = None
    db = None

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ###########################################################################
    #                             END OF YOUR CODE                            #
    ###########################################################################
    return dx, dw, db


In [ ]:
# Test the affine_backward function
np.random.seed(231)
x = np.random.randn(10, 2, 3)
w = np.random.randn(6, 5)
b = np.random.randn(5)
dout = np.random.randn(10, 5)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)

# The error should be around e-10 or less
print('Testing affine_backward function:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

## 1.3 "Sandwich" layers
There are some common patterns of layers that are frequently used in neural nets. For example, affine layers are frequently followed by a ReLU nonlinearity. To make these common patterns easy, we define convenience layers in the following.

In [ ]:
def affine_relu_forward(x, w, b):
    """
    Convenience layer that performs an affine transform followed by a ReLU

    Inputs:
    - x: Input to the affine layer
    - w, b: Weights for the affine layer

    Returns a tuple of:
    - out: Output from the ReLU
    - cache: Object to give to the backward pass
    """
    a, fc_cache = affine_forward(x, w, b)
    out, relu_cache = relu_forward(a)
    cache = (fc_cache, relu_cache)
    return out, cache


def affine_relu_backward(dout, cache):
    """
    Backward pass for the affine-relu convenience layer
    """
    fc_cache, relu_cache = cache
    da = relu_backward(dout, relu_cache)
    dx, dw, db = affine_backward(da, fc_cache)
    return dx, dw, db

Let's check numerically the gradient backward pass

In [ ]:
np.random.seed(231)
x = np.random.randn(2, 3, 4)
w = np.random.randn(12, 10)
b = np.random.randn(10)
dout = np.random.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

# Relative error should be around e-10 or less
print('Testing affine_relu_forward and affine_relu_backward:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

<a id="Softmax_classifier"></a>
# 2. Softmax classifier
The softmax classifier is widely used in multi-class classification probems. You've implemented a binary Logistic Regression classifier in the first assignment, the Softmax classifier is its generalization to multiple classes. Unlike hinge-loss classifiers which treat the outputs $f(x_i,W)$ as (uncalibrated and possibly difficult to interpret) scores for each class, the Softmax classifier gives a slightly more intuitive output (normalized class probabilities) and also has a probabilistic interpretation. In the Softmax classifier, the function mapping $f(x_i;W) = W x_i$ stays unchanged, but we now interpret these scores as the unnormalized log probabilities for each class and replace the hinge loss with a cross-entropy loss that has the form:

$ L_i = -log \Big(\frac{e^{f_{y_i}}}{\sum_j e^{f_j}}\Big)$ or equivalently $L_i =-f_{y_i} + log \sum\limits_j e^{f_j}$

where we are using the notation $f_j$ to mean the j-th element of the vector of class scores $f$. As before, the full loss for the dataset is the mean of $L_i$ over all training examples. The function $f_j(z) = \frac{e^{z_j}}{\sum_k e^{z_k}}$ is called the softmax function: It takes a vector of arbitrary real-valued scores (in $z$) and squashes it to a vector of values between zero and one that sum to one. The full cross-entropy loss that involves the softmax function might look scary if you’re seeing it for the first time but it is relatively easy to motivate. Two really good motivations from a probabilistic and an information theoretic perspective are given on the [cs231n webpage](https://cs231n.github.io/linear-classify/).

### Numerical stability
When you’re writing code for computing the Softmax function in practice, the intermediate terms $e^{f_{y_i}}$ and $\sum_j e^{f_j}$ may be very large due to the exponentials. Dividing large numbers can be numerically unstable, so it is important to use a normalization trick. Notice that if we multiply the top and bottom of the fraction by a constant C and push it into the sum, we get the following (mathematically equivalent) expression:

$\frac{e^{f_{y_i}}}{\sum_j e^{f_j}} = \frac{C e^{f_{y_i}}}{C \sum_j e^{f_j}} =  \frac{e^{f_{y_i}+log(C)}}{\sum_j e^{f_j + log(C)}}$

We are free to choose the value of $C$. This will not change any of the results, but we can use this value to improve the numerical stability of the computation. A common choice for $C$ is to set $log(C)=−max_j(f_j)$. This simply states that we should shift the values inside the vector $f$ so that the highest value is zero. In code:
```python
f = np.array([123, 456, 789]) # example with 3 classes and each having large scores
p = np.exp(f) / np.sum(np.exp(f)) # Bad: Numeric problem, potential blowup

# instead: first shift the values of f so that the highest number is 0:
f -= np.max(f) # f becomes [-666, -333, 0]
p = np.exp(f) / np.sum(np.exp(f)) # safe to do, gives the correct answer
```

### Task: Cross-entropy loss
In the following, implement the `cross_entropy_loss` and test your implementation against the numerically approximated gradient.
<a id="cross_entropy_loss"></a>

In [ ]:
def cross_entropy_loss(x, y):
    """
    Computes the loss and gradient for softmax classification.

    Inputs:
    - x: Input data, of shape (N, C) where x[i, j] is the score for the jth
      class for the ith input.
    - y: Vector of labels, of shape (N,) where y[i] is the label for x[i] and
      0 <= y[i] < C

    Returns a tuple of:
    - loss: Scalar giving the loss
    - dx: Gradient of the loss with respect to x
    """
    # Initialize the loss and gradient to zero.
    loss = 0.0
    dx = np.zeros_like(x)
    #############################################################################
    # TODO: Compute the softmax loss and its gradient using no explicit loops.  #
    # Store the loss in loss and the gradient in dx. If you are not careful     #
    # here, it is easy to run into numeric instability.                         #
    #############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    loss = None
    dx = None
    
    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    #############################################################################
    #                          END OF YOUR CODE                                 #
    #############################################################################
    return loss, dx

In [ ]:
np.random.seed(231)
num_classes, num_inputs = 10, 50
x = 0.001 * np.random.randn(num_inputs, num_classes)
y = np.random.randint(num_classes, size=num_inputs)

dx_num = eval_numerical_gradient(lambda x: cross_entropy_loss(x, y)[0], x, verbose=False)
loss, dx = cross_entropy_loss(x, y)

# Test cross_entropy_loss function. Loss should be close to 2.3 and dx error should be around e-8
print('\nTesting cross_entropy_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))

## 2.1 Two layer fully-connected Neural Network
Now we have to put all the elements together. We want to construct a two layer network class `TwoLayerNet`, therefore we need to initialize two sets of weights and biases, one set for each of the two layers. This needs to be implemented in the `__init__` method. In the `loss` method, the parameters are unpacked and then the forward pass is done. It computes all activations throughout the network, including the hypothesis $h_\theta(x)$. The hypothesis and the groundtruth values `y` are fed into the `cross_entropy_loss` which returns a loss value and the gradient $\frac{\partial L_\theta}{\partial a^{(3)}}$. Then the upstream derivatives `dx` and the partial derivatives with respect to the parameters of the last layer `dw` and `db` are calculated by the function `affine_relu_backward`. The upstream derivative `dx` is again fed into `affine_relu_backward` together with the cached variables from the first layer to calculate `dw` and `db` for the hidden layer. All of the gradients have to be stored into the dictionary `grads` and will be used by the `Solver` to update the parameters of the model.

![Network architecture](https://i.imgur.com/Gl5bdaa.png) 
<a id="TwoLayerNet"></a>

In [ ]:
class TwoLayerNet(object):
    """
    A two-layer fully-connected neural network with ReLU nonlinearity and
    cross_entropy_loss that uses a modular layer design. We assume an input dimension
    of D, a hidden dimension of H, and perform classification over C classes.

    The architecure should be affine - relu - affine - relu.

    Note that this class does not implement gradient descent; instead, it
    will interact with a separate Solver object that is responsible for running
    optimization.

    The learnable parameters of the model are stored in the dictionary
    self.params that maps parameter names to numpy arrays.
    """

    def __init__(
        self,
        input_dim=3 * 32 * 32,
        hidden_dim=100,
        num_classes=10,
        weight_scale=1e-2,
    ):
        """
        Initialize a new network.

        Inputs:
        - input_dim: An integer giving the size of the input
        - hidden_dim: An integer giving the size of the hidden layer
        - num_classes: An integer giving the number of classes to classify
        - weight_scale: Scalar giving the standard deviation for random
          initialization of the weights.
        """
        self.params = {}

        # Initialize the weights and biases of the two-layer net.
        self.params['W1'] = weight_scale * np.random.randn(input_dim, hidden_dim)
        self.params['b1'] = np.zeros(hidden_dim)
        self.params['W2'] = weight_scale * np.random.randn(hidden_dim, num_classes)
        self.params['b2'] = np.zeros(num_classes)


    def loss(self, X, y=None):
        """
        Compute loss and gradient for a minibatch of data.

        Inputs:
        - X: Array of input data of shape (N, d_1, ..., d_k)
        - y: Array of labels, of shape (N,). y[i] gives the label for X[i].

        Returns:
        If y is None, then run a test-time forward pass of the model and return:
        - scores: Array of shape (N, C) giving classification scores, where
          scores[i, c] is the classification score for X[i] and class c.

        If y is not None, then run a training-time forward and backward pass and
        return a tuple of:
        - loss: Scalar value giving the loss
        - grads: Dictionary with the same keys as self.params, mapping parameter
          names to gradients of the loss with respect to those parameters.
        """
        scores = None
        W1 = self.params['W1']
        b1 = self.params['b1']
        W2 = self.params['W2']
        b2 = self.params['b2']
        
        X2, relu_cache = affine_relu_forward(X, W1, b1)
        scores, relu2_cache = affine_relu_forward(X2, W2, b2)

        # If y is None then we are in test mode so just return scores
        if y is None:
            return scores
        
        loss, grads = 0, {}
        loss, softmax_grad = cross_entropy_loss(scores, y)
        # calculate gradient
        dx2, dw2, db2 = affine_relu_backward(softmax_grad, relu2_cache)
        dx, dw, db = affine_relu_backward(dx2, relu_cache)
        grads['W2'] = dw2 
        grads['b2'] = db2
        grads['W1'] = dw 
        grads['b1'] = db

        return loss, grads

For monitoring the training process we use Tensorboard. Tensorboard is a tool for visualizing data and models during and after training. It can be used to monitor loss and accuracy during training, as well as histograms of model parameters during training.

To use Tensorboard, first run the cell below. At first, you won't see any data in Tensorboard. Once you run the training cell, you need to refresh the Tensorboard cell to see the data.

In [ ]:
# Launch TensorBoard in Colab-style notebooks
log_dir = "logs"
%load_ext tensorboard
%tensorboard --logdir {log_dir}

## 2.2 Train the network using the pre-built solver

In [ ]:
# NOTE: Keep the hyperparameters fixed for the first training run,
# you can finetune afterwards.

# Set hyperparameters
learning_rate = 1e-3 # learning rate
hidden_dim = 100 # number of neurons in the hidden layer
epochs = 20 # number of training epochs

# build the network
model = TwoLayerNet(hidden_dim = hidden_dim)

# optimize
solver = Solver(
    model,
    data,
    optim_config={'learning_rate': learning_rate}, 
    lr_decay=0.95, # decay learning rate by this factor every epoch
    num_epochs=epochs,
    batch_size=100,
    print_every=100,
    log_dir=log_dir
)

solver.train()  

With the initial hyperparameters the overall validation accuracy of the classification model is around 45% to 50%. 
This is better than random guessing (~10% for 10 classes) but still not great.

Try tuning various training hyperparameters (such as learning rate, batch size, number of epochs, etc.) and see if you can improve the model performance.

## 2.3 Model evaluation
We evaluate the model on the test set and plot the confusion matrix below.
On the vertical axis are the true labels, on the horizontal axis are the predicted labels.
One row of the confusion matrix shows how many images (in %) of a true class were correctly classified and misclassified as another class.

In [ ]:


acc, pred = solver.check_accuracy(data['X_test'], data['y_test'], num_samples=10000)
y_pred = pred['y_pred']
print('#########################################')
print('Overall Accuracy on the test set: ', acc)
print('#########################################')

from sklearn.metrics import confusion_matrix
from seaborn import heatmap
# sns heatmap because plt.matshow has bugs with setting ticklabels

names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
conf_mat = confusion_matrix(y_true= data['y_test'], y_pred=y_pred)
conf_mat = conf_mat/1000
conf_mat = np.round(conf_mat,2)

fig, ax = plt.subplots(figsize=(10,10))
# Plot the heatmap
heatmap(conf_mat, cmap="Blues", annot=True, annot_kws={"size": 10}, xticklabels=names, yticklabels=names, ax=ax)
# Add y and x labels
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
plt.tight_layout()  
plt.show()

Confusion matrices are a useful way to visualize the performance of a classification model and get an insight into which classes the model is struggling with the most.

Reflect on the confusion matrix:
- Which class does the model classify most accurately?
- Which class does it struggle with the most?
- For the class with the lowest accuracy, which other class is it most often confused with?

### Browse test predictions
Below you can insert different image indices `img_idx` and get the associated test sample image and a bar chart showing the label predictions as well as the true label. 

In [ ]:
# Your test set consists of 10,000 examples,
# so integers in the intervall [0,9999] are valid.

img_idx = 42

def softmax(x):
    """Compute softmax values for each sets of scores in x."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

def show_prediction(image_idx):
    img = (np.transpose(data['X_test'][image_idx], [1,2,0])/255)+0.5
    scores = np.transpose(pred['scores'])
    scores = np.round(softmax(scores[image_idx]),2)
    gt = np.zeros(10)
    gt[data['y_test'][image_idx]]=1

    plt.figure(figsize = (2,2))
    plt.imshow(img, interpolation='lanczos')
    # Remove ticks
    ax = plt.gca()
    ax.set_xticks([])
    ax.set_yticks([])

    names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    x_axis = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(10,8))
    width = 0.5 
    rects1 = ax.bar(x_axis-width/2, gt, 0.5, label='Groundtruth')
    rects1 = ax.bar(x_axis+width/2, scores, 0.5, label='Prediction probability')
    ax.set_xticks(x_axis)
    ax.set_xticklabels(names)
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.show()
    return None

show_prediction(img_idx)

## Plot worst predictions

In practice, it is crucial to understand why a model fails. For our model, we can plot the "worst" test samples as confident wrong predictions: those where the model predicts the wrong class with high probability.

In [ ]:
scores = np.transpose(pred['scores'])
probs = softmax(scores)

predicted_classes = y_pred
predicted_confidence = probs[np.arange(len(predicted_classes)), predicted_classes]

y_true = data['y_test']
wrong_idx = np.flatnonzero(predicted_classes != y_true)

# sort only the wrong predictions by descending confidence
worst = wrong_idx[np.argsort(-predicted_confidence[wrong_idx])]
print("The 5 most confident wrong predictions are at img_idx:")
print(worst[:5])

Look at the worst few samples. What do you think, why is the model particularly bad at predicting these?

In [ ]:
# Plot worst images with true and predicted labels

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(30,20))
for i in range(49):
    idx = worst[i]
    true_label = data['y_test'][idx]
    pred_label = y_pred[idx]

    plt.subplot(7,7,i+1)
    img = np.clip((np.transpose(data['X_test'][idx], [1,2,0])/255)+0.5,0,1)
    plt.imshow(img, interpolation='lanczos')
    plt.title(f"True: {true_label} ({class_names[true_label]})\nPred: {pred_label} ({class_names[pred_label]})", fontsize=20)
    # Remove ticks
    ax = plt.gca()
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off')

plt.tight_layout()
plt.show()